# APL Logistics Delivery Performance Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib

## Load data

In [ ]:
df = pd.read_csv("./data/raw/APL_Logistics.csv", encoding="latin1", low_memory=False)
print("Dataset shape:", df.shape)
df.head()

In [ ]:
print("Columns:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\nDuplicate rows:", df.duplicated().sum())
print("Date columns:", [c for c in df.columns if "date" in c.lower()])

## Data cleaning and validation

In [ ]:
# Shipping day columns must be numeric and not negative
number_columns = ["Days for shipping (real)", "Days for shipment (scheduled)",
                  "Late_delivery_risk", "Sales", "Order Profit Per Order"]
for col in number_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

rows_before = len(df)
df = df.drop_duplicates()
df = df.dropna(subset=["Days for shipping (real)", "Days for shipment (scheduled)", "Late_delivery_risk"])
df = df[(df["Days for shipping (real)"] >= 0) & (df["Days for shipment (scheduled)"] >= 0)].copy()
print("Rows removed:", rows_before - len(df))

In [ ]:
# Standardize region, market, country and other text names (removes extra spaces like "South of  USA ")
text_columns = ["Order Region", "Order Country", "Order State", "Order City", "Market",
                "Shipping Mode", "Customer Segment", "Delivery Status", "Order Status"]
for col in text_columns:
    df[col] = df[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

print("Number of regions:", df["Order Region"].nunique())
print(sorted(df["Order Region"].unique()))

In [ ]:
# Shipping days per mode - checking that values make sense
df.groupby("Shipping Mode")[["Days for shipment (scheduled)", "Days for shipping (real)"]].agg(["min", "max", "mean"]).round(2)

## Delivery gap calculation

In [ ]:
df["Delay Gap"] = df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]

df["Delivery Timing"] = np.select(
    [df["Delay Gap"] > 0, df["Delay Gap"] < 0],
    ["Delayed", "Early"],
    default="On-time"
)
df[["Days for shipping (real)", "Days for shipment (scheduled)", "Delay Gap", "Delivery Timing", "Late_delivery_risk"]].head()

## Delivery Status vs Late_delivery_risk
Canceled shipments never reached the customer, so they are checked here and then removed from the delivery analysis.

In [ ]:
# How does the calculated timing match the Delivery Status column?
print(pd.crosstab(df["Delivery Status"], df["Delivery Timing"]))
print()
print(pd.crosstab(df["Delivery Status"], df["Late_delivery_risk"]))

In [ ]:
# Canceled orders have a delay gap but Late_delivery_risk is always 0 for them
canceled = df[df["Delivery Status"] == "Shipping canceled"]
print("Canceled shipments:", len(canceled))
print("Canceled with Delayed timing:", (canceled["Delivery Timing"] == "Delayed").sum())

df_all = df.copy()
df = df[df["Delivery Status"] != "Shipping canceled"].copy()
print("Rows used for delivery analysis:", len(df))

# After removing canceled orders, delayed timing and Late_delivery_risk should match exactly
print("Delayed == Late_delivery_risk for all rows:",
      ((df["Delivery Timing"] == "Delayed") == (df["Late_delivery_risk"] == 1)).all())

## Overall delivery performance (KPIs)

In [ ]:
total_orders = len(df)
on_time_rate = (df["Delay Gap"] <= 0).mean() * 100
delayed_rate = (df["Delivery Timing"] == "Delayed").mean() * 100
early_rate = (df["Delivery Timing"] == "Early").mean() * 100
exact_rate = (df["Delivery Timing"] == "On-time").mean() * 100
average_delay = df["Delay Gap"].mean()
average_delay_late = df[df["Delay Gap"] > 0]["Delay Gap"].mean()
late_risk_rate = df["Late_delivery_risk"].mean()

summary = pd.DataFrame({
    "Metric": [
        "Total orders analysed",
        "On-time delivery rate (on time or early, %)",
        "Exactly on schedule (%)",
        "Early delivery (%)",
        "Delayed delivery (%)",
        "Average delivery delay (days)",
        "Average delay of late orders (days)",
        "Late delivery risk ratio"
    ],
    "Value": [total_orders, round(on_time_rate, 2), round(exact_rate, 2), round(early_rate, 2),
              round(delayed_rate, 2), round(average_delay, 2), round(average_delay_late, 2),
              round(late_risk_rate, 3)]
})
summary

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))

df["Delivery Timing"].value_counts().reindex(["On-time", "Delayed", "Early"]).plot(kind="bar", ax=ax[0], color=["gold", "tomato", "seagreen"])
ax[0].set_title("Delivery timing")
ax[0].tick_params(axis="x", rotation=0)

df["Late_delivery_risk"].value_counts().sort_index().plot(kind="bar", ax=ax[1], color=["seagreen", "tomato"])
ax[1].set_title("Late_delivery_risk (0 = no, 1 = late)")
ax[1].tick_params(axis="x", rotation=0)

ax[2].hist(df["Delay Gap"], bins=np.arange(-2.5, 5.5, 1), edgecolor="black")
ax[2].set_title("Delay gap (days)")

plt.tight_layout()
plt.savefig("outputs/figures/01_delivery_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## Shipping mode efficiency

In [ ]:
shipping_summary = df.groupby("Shipping Mode").agg(
    Orders=("Delay Gap", "size"),
    Scheduled_Days=("Days for shipment (scheduled)", "mean"),
    Actual_Days=("Days for shipping (real)", "mean"),
    Average_Delay_Gap=("Delay Gap", "mean"),
    Delayed_Rate=("Delivery Timing", lambda x: (x == "Delayed").mean() * 100),
).reset_index()

shipping_summary["SLA_Compliance"] = 100 - shipping_summary["Delayed_Rate"]
# Efficiency index: mode delayed rate / overall delayed rate (above 1 = worse than average)
shipping_summary["Efficiency_Index"] = shipping_summary["Delayed_Rate"] / delayed_rate
shipping_summary = shipping_summary.round(2).sort_values("Delayed_Rate", ascending=False)
shipping_summary

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

shipping_summary.plot(x="Shipping Mode", y="Delayed_Rate", kind="bar", legend=False, ax=ax[0], color="tomato")
ax[0].set_title("Delayed rate (%)")
ax[0].set_xlabel("")
ax[0].tick_params(axis="x", rotation=30)

shipping_summary.plot(x="Shipping Mode", y="SLA_Compliance", kind="bar", legend=False, ax=ax[1], color="seagreen")
ax[1].set_title("SLA compliance (%)")
ax[1].set_xlabel("")
ax[1].tick_params(axis="x", rotation=30)

shipping_summary.plot(x="Shipping Mode", y=["Scheduled_Days", "Actual_Days"], kind="bar", ax=ax[2])
ax[2].set_title("Scheduled vs actual days")
ax[2].set_xlabel("")
ax[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("outputs/figures/02_shipping_mode_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Delay gap values for each mode (rows = mode, columns = delay gap in days)
pd.crosstab(df["Shipping Mode"], df["Delay Gap"])

## Regional and market diagnostics

In [ ]:
market_summary = df.groupby("Market").agg(
    Orders=("Delay Gap", "size"),
    Average_Delay_Gap=("Delay Gap", "mean"),
    Delayed_Rate=("Delivery Timing", lambda x: (x == "Delayed").mean() * 100)
).reset_index()
market_summary["Regional_Delay_Index"] = market_summary["Delayed_Rate"] / delayed_rate
market_summary.round(2).sort_values("Delayed_Rate", ascending=False)

In [ ]:
region_summary = df.groupby("Order Region").agg(
    Orders=("Delay Gap", "size"),
    Average_Delay_Gap=("Delay Gap", "mean"),
    Delayed_Rate=("Delivery Timing", lambda x: (x == "Delayed").mean() * 100)
).reset_index()
region_summary["Regional_Delay_Index"] = region_summary["Delayed_Rate"] / delayed_rate
region_summary = region_summary.round(2).sort_values("Delayed_Rate", ascending=False)

print("Delayed rate range across regions:", region_summary["Delayed_Rate"].min(), "to", region_summary["Delayed_Rate"].max())
region_summary

In [ ]:
# Regions use a different mix of shipping modes, so compare each region with what its mode mix would predict
df["Expected_Delay"] = df.groupby("Shipping Mode")["Late_delivery_risk"].transform("mean")
region_adjusted = df.groupby("Order Region").agg(
    Actual=("Late_delivery_risk", "mean"), Expected=("Expected_Delay", "mean"))
region_adjusted["Adjusted_Index"] = region_adjusted["Actual"] / region_adjusted["Expected"]
region_adjusted.round(3).sort_values("Adjusted_Index", ascending=False).head(5)

In [ ]:
# Countries with at least 500 orders
country_summary = df.groupby("Order Country").agg(
    Orders=("Delay Gap", "size"),
    Average_Delay_Gap=("Delay Gap", "mean"),
    Delayed_Rate=("Delivery Timing", lambda x: (x == "Delayed").mean() * 100)
).reset_index()
country_summary = country_summary[country_summary["Orders"] >= 500].round(2)
print("Highest delayed rate")
print(country_summary.sort_values("Delayed_Rate", ascending=False).head(10).to_string(index=False))
print("\nLowest delayed rate")
print(country_summary.sort_values("Delayed_Rate").head(5).to_string(index=False))

In [ ]:
# Heatmap: delayed rate by region and shipping mode
heat = df.pivot_table(index="Order Region", columns="Shipping Mode", values="Late_delivery_risk", aggfunc="mean") * 100

plt.figure(figsize=(8, 9))
plt.imshow(heat, cmap="Reds", aspect="auto")
plt.colorbar(label="Delayed rate (%)")
plt.xticks(range(len(heat.columns)), heat.columns, rotation=30)
plt.yticks(range(len(heat.index)), heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        plt.text(j, i, round(heat.iloc[i, j]), ha="center", va="center", fontsize=8,
                 color="white" if heat.iloc[i, j] > 70 else "black")
plt.title("Delayed rate (%) by region and shipping mode")
plt.tight_layout()
plt.savefig("outputs/figures/03_heatmap_region_vs_shipping_mode.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Heatmap: market by shipping mode
heat2 = df.pivot_table(index="Market", columns="Shipping Mode", values="Late_delivery_risk", aggfunc="mean") * 100

plt.figure(figsize=(7, 3.5))
plt.imshow(heat2, cmap="Reds", aspect="auto")
plt.colorbar(label="Delayed rate (%)")
plt.xticks(range(len(heat2.columns)), heat2.columns, rotation=30)
plt.yticks(range(len(heat2.index)), heat2.index)
for i in range(heat2.shape[0]):
    for j in range(heat2.shape[1]):
        plt.text(j, i, round(heat2.iloc[i, j]), ha="center", va="center",
                 color="white" if heat2.iloc[i, j] > 70 else "black")
plt.title("Delayed rate (%) by market and shipping mode")
plt.tight_layout()
plt.savefig("outputs/figures/04_heatmap_market_vs_shipping_mode.png", dpi=150, bbox_inches="tight")
plt.show()

## Customer segment impact

In [ ]:
segment_summary = df.groupby("Customer Segment").agg(
    Orders=("Delay Gap", "size"),
    Average_Delay_Gap=("Delay Gap", "mean"),
    Delayed_Rate=("Delivery Timing", lambda x: (x == "Delayed").mean() * 100),
    Late_Orders=("Late_delivery_risk", "sum")
).reset_index()

# SLA risk exposure: share of all late orders compared with share of all orders
segment_summary["Share_of_Orders_%"] = segment_summary["Orders"] / segment_summary["Orders"].sum() * 100
segment_summary["Share_of_Late_Orders_%"] = segment_summary["Late_Orders"] / segment_summary["Late_Orders"].sum() * 100
segment_summary.round(2)

In [ ]:
# Does the shipping mode mix differ by segment? (delayed rate %, rows = segment)
(df.pivot_table(index="Customer Segment", columns="Shipping Mode", values="Late_delivery_risk", aggfunc="mean") * 100).round(2)

In [ ]:
# Sales value of delayed orders by segment
late_sales = df[df["Delivery Timing"] == "Delayed"].groupby("Customer Segment")["Sales"].sum().round(0)
late_sales

## Sales and profit overview

In [ ]:
print("Total sales:", round(df["Sales"].sum(), 2))
print("Total order profit:", round(df["Order Profit Per Order"].sum(), 2))
print("Sales on delayed orders:", round(df[df["Delivery Timing"] == "Delayed"]["Sales"].sum(), 2))
print("Average profit per order - delayed:", round(df[df["Delivery Timing"] == "Delayed"]["Order Profit Per Order"].mean(), 2))
print("Average profit per order - not delayed:", round(df[df["Delivery Timing"] != "Delayed"]["Order Profit Per Order"].mean(), 2))

## Predicting late delivery (ML model)
Goal: predict Late_delivery_risk before shipping. Delivery Status, real shipping days and Delay Gap are NOT used as features because they are only known after delivery and would leak the answer.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Only information known at order time
feature_columns = ["Type", "Shipping Mode", "Days for shipment (scheduled)", "Market", "Order Region",
                   "Customer Segment", "Category Name", "Department Name", "Order Item Quantity",
                   "Order Item Discount Rate", "Order Item Product Price", "Sales"]

X = pd.get_dummies(df[feature_columns], drop_first=True).astype(float)
y = df["Late_delivery_risk"]
print("Features after encoding:", X.shape[1])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
# Baselines to beat: (1) always predict the most common class, (2) predict from shipping mode only
majority_pred = np.full(len(y_test), y_train.mode()[0])
print("Majority class accuracy:", round(accuracy_score(y_test, majority_pred), 4))

train_data = df.loc[X_train.index]
mode_rate = train_data.groupby("Shipping Mode")["Late_delivery_risk"].mean()
mode_pred = (df.loc[X_test.index, "Shipping Mode"].map(mode_rate) > 0.5).astype(int)
print("Shipping mode only accuracy:", round(accuracy_score(y_test, mode_pred), 4))
print("Shipping mode only AUC:", round(roc_auc_score(y_test, df.loc[X_test.index, "Shipping Mode"].map(mode_rate)), 4))

In [ ]:
# Model 1: Logistic Regression (features are scaled so it converges)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)
log_pred = log_model.predict(X_test_scaled)
log_prob = log_model.predict_proba(X_test_scaled)[:, 1]

# Model 2: Random Forest
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [accuracy_score(y_test, log_pred), accuracy_score(y_test, rf_pred)],
    "Precision": [precision_score(y_test, log_pred), precision_score(y_test, rf_pred)],
    "Recall": [recall_score(y_test, log_pred), recall_score(y_test, rf_pred)],
    "F1": [f1_score(y_test, log_pred), f1_score(y_test, rf_pred)],
    "ROC_AUC": [roc_auc_score(y_test, log_prob), roc_auc_score(y_test, rf_prob)]
}).round(4)
results

In [ ]:
# Confusion matrix of the Random Forest
cm = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(4, 3.5))
plt.imshow(cm, cmap="Blues")
plt.xticks([0, 1], ["Not late", "Late"])
plt.yticks([0, 1], ["Not late", "Late"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.title("Random Forest confusion matrix")
plt.tight_layout()
plt.savefig("outputs/figures/05_confusion_matrix_random_forest.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Which features matter most?
importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(7, 4))
importance[::-1].plot(kind="barh")
plt.title("Top 10 feature importances (Random Forest)")
plt.tight_layout()
plt.savefig("outputs/figures/06_feature_importance_random_forest.png", dpi=150, bbox_inches="tight")
plt.show()
importance.round(4)

In [ ]:
# Accuracy by shipping mode - where does the model help?
test_data = df.loc[X_test.index].copy()
test_data["Correct"] = (rf_pred == y_test.values)
test_data.groupby("Shipping Mode")["Correct"].mean().round(4)

In [ ]:
# Save the trained models, the scaler and the feature names for the Streamlit app
joblib.dump(log_model, "models/logistic_regression.joblib")
joblib.dump(rf_model, "models/random_forest.joblib")
joblib.dump(scaler, "models/scaler.joblib")
joblib.dump(list(X.columns), "models/feature_columns.joblib")

# Save the test results too, including the simple shipping mode rule for comparison
mode_prob = df.loc[X_test.index, "Shipping Mode"].map(mode_rate)
baseline = pd.DataFrame({
    "Model": ["Shipping mode rule (baseline)"],
    "Accuracy": [accuracy_score(y_test, mode_prob > 0.5)],
    "Precision": [precision_score(y_test, mode_prob > 0.5)],
    "Recall": [recall_score(y_test, mode_prob > 0.5)],
    "F1": [f1_score(y_test, mode_prob > 0.5)],
    "ROC_AUC": [roc_auc_score(y_test, mode_prob)]
}).round(4)
pd.concat([results, baseline], ignore_index=True).to_csv("models/model_results.csv", index=False)

print("Saved files:", sorted(os.listdir("models")))
print("Saved figures:", sorted(os.listdir("./outputs/figures/")))

## Export

In [ ]:
df.to_csv("./data/processed/APL_Logistics_cleaned.csv", index=False)
shipping_summary.to_csv("./outputs/tables/shipping_mode_summary.csv", index=False)
region_summary.to_csv("./outputs/tables/regional_summary.csv", index=False)
market_summary.to_csv("./outputs/tables/market_summary.csv", index=False)
segment_summary.to_csv("./outputs/tables/customer_segment_summary.csv", index=False)
print("Analysis completed. Summary files were saved.")